In [1]:
# Assignment 2: MapReduce grade-category counts
# Calculate for all the subjects.

from collections import defaultdict
import glob
from pathlib import Path

data_files = glob.glob("*.txt")
REQUESTED_CATEGORIES = ["S", "A", "B", "C", "D", "E", "F"]
# The source uses I (incomplete/absent), while the assignment asks for F.
SOURCE_TO_REQUESTED = {"I": "F"}

In [2]:
def mapper(records):
    """Emit one (category, 1) pair for each student."""
    for record in records:
        category = SOURCE_TO_REQUESTED.get(record["grade"], record["grade"])
        if category in REQUESTED_CATEGORIES:
            yield category, 1

def reducer(mapped_pairs):
    """Aggregate all values belonging to the same category."""
    totals = defaultdict(int)
    for category, count in mapped_pairs:
        totals[category] += count
    return totals

In [3]:
for data_path in data_files:
    text = Path(data_path).read_text(encoding="utf-8", errors="replace")
    records = []
    for line in text.splitlines():
        fields = line.split()
        if len(fields) >= 5 and fields[0].isdigit() and fields[-1].isalpha():
            records.append({"regno": fields[0], "grade": fields[-1].upper()})
    
    mapped_pairs = list(mapper(records))
    reduced_counts = reducer(mapped_pairs)
    count_table = {
        category: reduced_counts.get(category, 0)
        for category in REQUESTED_CATEGORIES
    }
    
    print(f"--- Results for {data_path} ---")
    print(f"Students mapped: {len(mapped_pairs)}")
    for category, count in count_table.items():
        print(f"{category}: {count}")
    print("\n")

--- Results for CD202A5.txt ---
Students mapped: 58
S: 0
A: 3
B: 10
C: 19
D: 19
E: 1
F: 6


--- Results for CD203A4.txt ---
Students mapped: 54
S: 4
A: 13
B: 10
C: 13
D: 13
E: 1
F: 0


--- Results for CD204A1.txt ---
Students mapped: 57
S: 0
A: 4
B: 15
C: 14
D: 11
E: 9
F: 4


--- Results for CD204A4.txt ---
Students mapped: 55
S: 6
A: 13
B: 15
C: 11
D: 8
E: 2
F: 0


--- Results for CD205A1.txt ---
Students mapped: 56
S: 3
A: 10
B: 17
C: 8
D: 10
E: 5
F: 3


--- Results for CD207A3.txt ---
Students mapped: 55
S: 1
A: 2
B: 7
C: 19
D: 12
E: 8
F: 6


--- Results for GN201B1.txt ---
Students mapped: 433
S: 5
A: 43
B: 123
C: 143
D: 76
E: 19
F: 24


--- Results for MA206B1.txt ---
Students mapped: 356
S: 20
A: 39
B: 44
C: 44
D: 62
E: 46
F: 101


